# Sentinel-1 and Sentinel-2 parcel zonal statistics

Configure and run `OpenEOJobManagerZonalStats` through the multi-user scheduler. openEO creates sensor-preprocessed physical-band NetCDF cubes per spatial batch. IQR cleaning, null filling, pixel indices, and parcel statistics run locally, producing one GeoParquet row per parcel with dated features and a separate annual summary. See [the detailed workflow](../../../docs/data_preparation/openeo/WORKFLOW.md).

In [16]:
from pathlib import Path
import os
import shutil
from IPython.display import display
import geopandas as gpd

from data_preparation.parcel_stats import estimate_utm_epsg_from_parcels
from data_preparation.parcel_stats.job_manager import compute_tile_buffer_metres
from data_preparation.parcel_stats.multiuser import (
    load_openeo_users_from_db,
    merge_and_save_results,
    run_parallel_extractions,
)
from shared.io import read_data, write_data, file_exists
from shared.spatial_statistics import spatial_clustering_using_buffer
import shared.geometry as geom_l

from data_preparation.features import (
    append_nearest_lake_features,
    append_parcel_elevation,
    append_soil_feature,
    create_dem_mosaic,
    find_dem_tiles,
    validate_projected_metric_crs,
)

In [17]:
class StopExecution(Exception):
    """Stops the notebook run silently, without a traceback."""
    def _render_traceback_(self):
        return []

## Inputs

Processing order:

`Sentinel-2 L2A`
`acquisition cloud mask`
`optional temporal reduction, or retain acquisitions`
`per-band, per-time-slice raster IQR outlier removal when enabled`
`local null filling`
`local pixel indices from cleaned physical bands`
`parcel statistics`
`one-row-per-parcel time-series pivot`
`ML-ready GeoParquet`

Temporal filling is past-only so later observations cannot leak into earlier feature periods.

In [18]:
# ============================================================================
# STEP 1: Read data and re-project
# ============================================================================
# Read the data and re-project to the desired EPSG

WORK_PATH = Path("C:/work_sat")
RUN_IDENTIFIER = "central_greece"

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "central_greece_prediction.parquet"
gdf_parcels = read_data(str(PARCELS_PATH))
print("Initial parcels data:")
display(gdf_parcels.head())
display(gdf_parcels.columns)

keep_columns = ["parcel_id", "crop_code", "crop_label", "pred_crop_code", "pred_crop_label", "pred_probability", "set_type",
                "svm_is_outlier", "geometry"]
gdf_parcels = gdf_parcels[keep_columns].copy()
display(gdf_parcels.head())
print(f"Parcels before dropping duplicate geometries: {len(gdf_parcels)}")
gdf_parcels = gdf_parcels.drop_duplicates(subset=["geometry"]).copy()
print(f"Parcels after dropping duplicate geometries: {len(gdf_parcels)}")

gdf_parcels = geom_l.return_valid_geometries(gdf_parcels)
gdf_parcels = geom_l.convert_multipolygons_to_polygons(gdf_parcels)
print(f"Number of parcels after ensuring valid geometries and converting multipolygons to polygons: {len(gdf_parcels)}")

display(gdf_parcels)

Reading file: C:/work_sat/central_greece/central_greece_prediction.parquet
Initial parcels data:


,parcel_id,parcel_multiyear_id,year,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,pred_correct,set_type,...,cocultivation,sowing,zone_id,s2_tile_id,cartographic_basemap,area_ha,shape_area,shape_length,pix_count,geometry
0,753,10251480,2024,108,nut_trees,108,nut_trees,0.441608,True,train,...,False,False,2,24,3963979322004,0.961193,9611.928421,529.575797,32.0,"MULTIPOLYGON (((396871.63 4397265.407, 396875...."
1,136423,7588665,2024,101,olive_groves,101,olive_groves,0.723092,True,test,...,False,False,2,24,3764049003002,0.401124,4011.240489,276.755568,9.0,"MULTIPOLYGON (((376861.31 4403934.534, 376851...."
2,204783,6231558,2024,101,olive_groves,101,olive_groves,0.710824,True,test,...,False,False,2,24,3774000994007,0.467767,4677.673439,404.692436,8.0,"MULTIPOLYGON (((377236.773 4401113.632, 377254..."
3,245497,10770682,2024,101,olive_groves,129,fallow,0.545589,False,test,...,False,False,2,24,3814232856003,0.353199,3531.994102,238.988690,9.0,"MULTIPOLYGON (((381314.955 4423763.509, 381299..."
4,797263,9353904,2024,102,grape_vines,102,grape_vines,0.750356,True,test,...,False,False,2,24,3603984176030,0.383018,3830.181429,263.445246,9.0,"MULTIPOLYGON (((360493.572 4398483.286, 360554..."


Index(['parcel_id', 'parcel_multiyear_id', 'year', 'crop_code', 'crop_label',
       'pred_crop_code', 'pred_crop_label', 'pred_probability', 'pred_correct',
       'set_type', 'in_holdout', 'holdout_full', 'svm_is_outlier',
       'svm_outlier_full', 'lpis_normal', 'has_rfv', 'rfv_true',
       'declared_crop_type_id', 'declared_crop_variation_id', 'cocultivation',
       'sowing', 'zone_id', 's2_tile_id', 'cartographic_basemap', 'area_ha',
       'shape_area', 'shape_length', 'pix_count', 'geometry'],
      dtype='object')

,parcel_id,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,set_type,svm_is_outlier,geometry
0,753,108,nut_trees,108,nut_trees,0.441608,train,False,"MULTIPOLYGON (((396871.63 4397265.407, 396875...."
1,136423,101,olive_groves,101,olive_groves,0.723092,test,False,"MULTIPOLYGON (((376861.31 4403934.534, 376851...."
2,204783,101,olive_groves,101,olive_groves,0.710824,test,False,"MULTIPOLYGON (((377236.773 4401113.632, 377254..."
3,245497,101,olive_groves,129,fallow,0.545589,test,False,"MULTIPOLYGON (((381314.955 4423763.509, 381299..."
4,797263,102,grape_vines,102,grape_vines,0.750356,test,False,"MULTIPOLYGON (((360493.572 4398483.286, 360554..."


Parcels before dropping duplicate geometries: 23073
Parcels after dropping duplicate geometries: 23073
Number of parcels after ensuring valid geometries and converting multipolygons to polygons: 23073


,parcel_id,crop_code,crop_label,pred_crop_code,pred_crop_label,pred_probability,set_type,svm_is_outlier,geometry
0,753,108,nut_trees,108,nut_trees,0.441608,train,False,"POLYGON ((396871.63 4397265.407, 396875.582 43..."
1,136423,101,olive_groves,101,olive_groves,0.723092,test,False,"POLYGON ((376861.31 4403934.534, 376851.653 44..."
2,204783,101,olive_groves,101,olive_groves,0.710824,test,False,"POLYGON ((377236.773 4401113.632, 377254.798 4..."
3,245497,101,olive_groves,129,fallow,0.545589,test,False,"POLYGON ((381314.955 4423763.509, 381299.158 4..."
4,797263,102,grape_vines,102,grape_vines,0.750356,test,False,"POLYGON ((360493.572 4398483.286, 360554.752 4..."
...,...,...,...,...,...,...,...,...,...
23068,294878979,130,pastures,130,pastures,0.564301,test,False,"POLYGON ((366748.995 4402517.412, 366748.988 4..."
23069,294878980,130,pastures,130,pastures,0.663258,train,False,"POLYGON ((366547.416 4403202.695, 366709.21 44..."
23070,294878981,101,olive_groves,130,pastures,0.341065,test,True,"POLYGON ((366561.698 4402366.616, 366577.042 4..."
23071,294878983,130,pastures,130,pastures,0.650957,train,False,"POLYGON ((367460.882 4403400.962, 367468.411 4..."


In [19]:
train_df = gdf_parcels[gdf_parcels["set_type"] == "train"].copy()
test_df = gdf_parcels[gdf_parcels["set_type"] == "test"].copy()

train_accuracy = (train_df["crop_code"] == train_df["pred_crop_code"]).mean()
print(f"Train accuracy: {train_accuracy:.4f}")
test_accuracy = (test_df["crop_code"] == test_df["pred_crop_code"]).mean()
print(f"Test accuracy: {test_accuracy:.4f}")

total_accuracy = (gdf_parcels["crop_code"] == gdf_parcels["pred_crop_code"]).mean()
print(f"Total accuracy: {total_accuracy:.4f}")

Train accuracy: 0.8526
Test accuracy: 0.6864
Total accuracy: 0.7528


In [ ]:
print(gdf_parcels.crs)
WORKING_EPSG = gdf_parcels.crs.to_epsg()
# WORKING_EPSG = estimate_utm_epsg_from_parcels(gdf_parcels)
# WORKING_EPSG

In [ ]:
# ============================================================================
# Define the class column and parcel ID column names.
# These are used in the extraction process to identify the class of each parcel
# and its unique identifier.
# ============================================================================
class_column = "crop_label"
parcel_id_column = "parcel_id"

In [ ]:
label_counts = gdf_parcels[class_column].value_counts()
rare_labels = label_counts[label_counts < 10].index
gdf_parcels.loc[gdf_parcels[class_column].isin(rare_labels), class_column] = "9999"
print(f"Number of unique labels after grouping rare labels: {len(gdf_parcels[class_column].value_counts())}")
print(gdf_parcels[class_column].value_counts())

cond = gdf_parcels[class_column] == "9999"
print(50*"-")
print(f"Excluded parcels with label '9999': {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()
print(f"Selected parcels after excluding label '9999': {len(gdf_parcels)}")

print(50*"-")
exclude_categories = ["tobacco", "aromatic", "grass_fodders", "horticultural_fall"]
cond = gdf_parcels[class_column].isin(exclude_categories)
print(f"Excluded parcels with label {exclude_categories}: {cond.sum()} out of {len(gdf_parcels)}")
gdf_parcels = gdf_parcels[~cond].copy()
print(f"Selected parcels after excluding label {exclude_categories}: {len(gdf_parcels)}")
print(gdf_parcels[class_column].value_counts())

PARCELS_PATH = WORK_PATH / RUN_IDENTIFIER / "cleaned_grouped_parcels.gpkg"
write_data(gdf_parcels, str(PARCELS_PATH))

# Add Soil

In [ ]:
SOIL_PATH = Path("E:/Eleni/work_dir/Agro_Soil_Clean.gpkg")
SOIL_LAYER = "agrosoil"

gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

if file_exists(str(SOIL_PATH)):
    gdf_parcels = append_soil_feature(
        parcels_path=PARCELS_PATH,
        soil_path=SOIL_PATH,
        output_path=PARCELS_PATH,
        soil_layer=SOIL_LAYER,
        attribute_column="GAIOIKANOT",
        overwrite=True,
    )
    assert gdf_parcels.crs == WORKING_EPSG

write_data(gdf_parcels, str(PARCELS_PATH))

# Add cluster, only if the number of parcels is greater than 2, otherwise skip this step

In [ ]:
# Reuse saved cluster labels; generate them only when the input has none.
CLUSTER_FIELD = "cluster"
if CLUSTER_FIELD not in gdf_parcels.columns:
    gdf_grouped = spatial_clustering_using_buffer(
        gdf_parcels.to_crs(epsg=WORKING_EPSG),
        parcel_id_column,
        min_cluster_distance_in_m=5_000,
        field_cluster=CLUSTER_FIELD,
        plot_results=False,
    )
    if gdf_grouped[CLUSTER_FIELD].isna().any():
        print("Every parcel must have a cluster label before partitioning.")
        raise StopExecution
    if len(gdf_grouped) != len(gdf_parcels):
        print("The number of parcels after clustering does not match the original count.")
        raise StopExecution
    if len(gdf_grouped[CLUSTER_FIELD].unique()) < 2:
        print("Clustering resulted in fewer than 2 unique clusters, which is insufficient for partitioning.")
        raise StopExecution
    gdf_parcels = gdf_grouped.copy()
    write_data(gdf_parcels, str(PARCELS_PATH))

# Add Elevation

In [ ]:
DEM_DIR = WORK_PATH / RUN_IDENTIFIER / "dem"
DEM_DIR.mkdir(parents=True, exist_ok=True)
MOSAIC_PATH = DEM_DIR / "dem_mosaic.tif"

EPSG_CODE = 2100  # Greek Grid (metric)
# False uses pixels whose centers fall inside each parcel. Set True for very small parcels.
ALL_TOUCHED = False

In [ ]:
def_files_path = "G:/Dem/ESA_10"
gdf_dem_tiles_path = os.path.join(def_files_path, "esa_tiles_dem_10m.gpkg")
gdf_dem_tiles = read_data(gdf_dem_tiles_path)
gdf_dem_tiles = gdf_dem_tiles.to_crs(epsg=EPSG_CODE)
tiles_to_copy = gdf_dem_tiles[gdf_dem_tiles.geometry.intersects(gdf_parcels.union_all())]
display(tiles_to_copy)
for _, tile in tiles_to_copy.iterrows():
    print(f"Processing tile: {tile}")
    src_tile_path = os.path.join(def_files_path, tile.file_name)
    dst_tile_path = os.path.join(DEM_DIR, tile.file_name)
    if not os.path.isfile(dst_tile_path):
        print(f"Copying {src_tile_path} to {dst_tile_path}")
        shutil.copy(src_tile_path, dst_tile_path)

dem_tiles = find_dem_tiles(DEM_DIR, MOSAIC_PATH)
# Reuses a valid existing mosaic. Set overwrite=True to rebuild it.
MOSAIC_PATH = create_dem_mosaic(dem_tiles, MOSAIC_PATH, overwrite=False)

In [ ]:
if file_exists(str(MOSAIC_PATH)):
    parcels_with_elevation = append_parcel_elevation(
        parcels_path=PARCELS_PATH,
        mosaic_path=MOSAIC_PATH,
        output_path=PARCELS_PATH,
        all_touched=ALL_TOUCHED,
        overwrite=True,
    )
    assert parcels_with_elevation.crs == WORKING_EPSG
    gdf_parcels = parcels_with_elevation.copy()
    write_data(gdf_parcels, str(PARCELS_PATH))

## Multi-user extraction planning

Accounts are loaded from PostgreSQL using `load_openeo_users_from_db(app_user_id=APP_USER_ID)`. Set the owner ID
in the next cell or provide `APP_USER_ID` in the kernel environment. Install the `database` and `credentials` extras.
Provide `DB_NAME` and `POSTGRES_*` in the kernel environment. The account cell reads the web app's
`SATELLITES_CREDENTIAL_KEY` from `.env` in the working directory or its parents, without changing database settings.
Install `python-dotenv` in the notebook kernel if needed (`%pip install python-dotenv`); never print the key.
`TBL_APP_USERS` and `TBL_EO_USERS` default to `public.app_users` and `public.eo_users`. `TBL_USERS` is ignored.
Only the active owner's accounts are read; passwords are decrypted in memory. Do not print or save credentials.
Accounts are sorted by username. Explicitly restore the original account order when resuming legacy jobs
without saved owners or using partition scheduling. See [database account setup](../../../docs/data_preparation/openeo/CREDENTIALS.md).

Create one spatial partition per cluster, then plan all tile batches before
submission. A shared queue lets every account process batches from any partition. With `fit_tiles_to_parcels=True`, a cube covers the
group's complete bounding box plus `TILE_BUFFER_METRES` on every side:

- width = east - west + 2 * buffer
- height = north - south + 2 * buffer

Only if either dimension exceeds `TILE_SIZE_METRES` does the planner split the
group spatially, then fit and check each subgroup in the same way. Parcels stay
whole and belong to exactly one job. A single parcel that exceeds the limit with
its buffer raises an error before submission.

A large partition can use all available accounts at once. Each account stays
within OPENEO_JOBS_PER_USER. Existing cubes are reused; previously submitted
jobs resume on their owning account. Keep the original account order when
first resuming a legacy run, which did not record account names.

Partition outputs and the merge step stay the same. Local statistics run after
the shared download queue completes, using LOCAL_BATCH_WORKERS per active
partition (up to one active partition per account).


In [ ]:
# ============================================================================
# STEP 2: Load accounts for the shared batch queue
# ============================================================================
# Accounts share unsubmitted batches from all cluster partitions.

from dotenv import dotenv_values, find_dotenv

# Refresh only the encryption key; keep the notebook's existing database connection settings.
credential_env_file = find_dotenv(usecwd=True)
if credential_env_file:
    credential_key = dotenv_values(credential_env_file).get("SATELLITES_CREDENTIAL_KEY")
    if credential_key is not None:
        os.environ["SATELLITES_CREDENTIAL_KEY"] = credential_key
    del credential_key

OPENEO_JOBS_PER_USER = 2  # This is default by openEO
APP_USER_ID = 3  # Selected app_users owner; use None to read APP_USER_ID from the kernel environment.
if APP_USER_ID is None:
    owner_setting = os.getenv("APP_USER_ID", "").strip()
    if not owner_setting.isdecimal() or int(owner_setting) <= 0:
        raise ValueError("Error: Set APP_USER_ID to your positive app_users.app_user_id before loading accounts.")
    APP_USER_ID = int(owner_setting)
users_list = load_openeo_users_from_db(app_user_id=APP_USER_ID)
print(f"Loaded {len(users_list)} openEO user(s):")
for i, (user, _) in enumerate(users_list, 1):
    print(f"  User {i}: {user}")

In [ ]:
# Keep each cluster together; the existing tile planner handles its size limit.
spatial_parts = [
    part.copy()
    for _, part in gdf_grouped.groupby(CLUSTER_FIELD, sort=True, observed=True)
]
assert sum(len(part) for part in spatial_parts) == len(gdf_grouped)

print(f"Created {len(spatial_parts)} cluster partitions:")
for i, part in enumerate(spatial_parts, 1):
    print(f"  Partition {i}: {part[CLUSTER_FIELD].iloc[0]} - {len(part)} parcels")


In [ ]:
# Maximum requested cube width AND height, including the buffer.
# Each group gets its own fitted extent; only oversized groups are split.
TILE_SIZE_METRES = 20_000
TILE_BUFFER_METRES = compute_tile_buffer_metres(gdf_grouped, WORKING_EPSG)

print(f"Maximum buffered cube width/height: {TILE_SIZE_METRES:,} m")
print(f"Buffer on each side: {TILE_BUFFER_METRES:,} m")


In [ ]:
# ============================================================================
# Confirm the planned global remote concurrency.
# ============================================================================
TOTAL_REMOTE_JOB_CAPACITY = len(users_list) * OPENEO_JOBS_PER_USER
print(
    f"Remote capacity: {len(users_list)} users x {OPENEO_JOBS_PER_USER} jobs "
    f"= {TOTAL_REMOTE_JOB_CAPACITY} simultaneous jobs"
)
